# E3 — Visualización 3D de reconstrucciones (matplotlib)

Usa matplotlib — las figuras se guardan como PNG en el notebook (sin pantalla en blanco).

**Por cada muestra, 4 vistas:**
1. **Entrada** — solo la pieza rota (rojo)
2. **Overlay roto + GT** — qué falta por reconstruir (rojo + azul transparente)
3. **Overlay predicción + GT** — calidad de la reconstrucción (verde + azul transparente)
4. **Error por punto** — la predicción coloreada por distancia al GT (azul=bueno, rojo=error)

**Cambia solo la Celda 3.**

In [ ]:
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
import os, subprocess
from getpass import getpass

if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr','/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')
else:
    print('[OK] PoinTr')

REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub: ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',REPO_DIR,'-q'], capture_output=True)
    del token
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)
subprocess.run(['pip','install','timm','easydict','pyyaml','--quiet'])
print('[OK]')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELDA 3 — CONFIGURACION
# ══════════════════════════════════════════════════════════════

VERSION = 'v5_fb_obj'   # modelo a cargar

DATASETS_VERSION = {
    'v5_fb_obj':      ['fb', 'obj'],
    'v5_obj':         ['obj'],
    'v5_all':         ['fb', 'obj', 'sn'],
    'v4_pointr_fbv2': ['fb'],
}
_partes = DATASETS_VERSION.get(VERSION, ['fb', 'obj'])

N_MOSTRAR  = 8      # muestras a visualizar (None = todas)
ALEATORIO  = True
ANGULO_AZ  = 30     # azimuth de la vista 3D (gira si los objetos se ven de lado)
ANGULO_EL  = 20     # elevación

print(f'Modelo: {VERSION} | Datasets: {_partes} | Muestras: {N_MOSTRAR}')

In [ ]:
# ── Cargar modelo ──────────────────────────────────────────────
import sys, os, types, glob as _glob
from pathlib import Path
import torch, torch.nn as nn
import numpy as np

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
DRIVE = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
BASE_GENERAL = f'{DRIVE}/Datos_E2_E3/General'

_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]

for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
        if new!=src: open(fp,'w',encoding='utf-8').write(new)
    except: pass

def _force(n,a):
    m=types.ModuleType(n)
    for k,v in a.items(): setattr(m,k,v)
    sys.modules[n]=m
def _inject(n,a):
    if n not in sys.modules: _force(n,a)
def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
class _CL1(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
class _CL2(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return ((d1**2).mean()+(d2**2).mean())/2
class _CPM(nn.Module):
    def forward(self,a,b): return torch.cdist(a.contiguous(),b.contiguous(),p=2).min(2).values.mean()
_ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL2,'ChamferDistanceL1_PM':_CPM,'chamfer_3DDist':_cr}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']: _force(n,_ch)

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np__):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np__,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np__):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1); dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu

class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})
def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']: _inject(pfx+base,attrs)

from easydict import EasyDict
ckpt_path = f'E3/checkpoints_pointr_{VERSION}/best.pt'
if not Path(ckpt_path).exists():
    ckpt_path = f'{BASE_E3}/modelos/{VERSION}/best.pt'
ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Checkpoint: {VERSION} epoca {ck["epoch"]}')
try:
    from models.build import build_model_from_cfg; model=build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr; model=PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict'])
model = model.to(device); model.eval()
print(f'[OK] {sum(p.numel() for p in model.parameters()):,} params')

In [ ]:
# ── Cargar test set ────────────────────────────────────────────
import subprocess, random
from pathlib import Path

FUENTES = {
    'fb':  (f'{BASE_GENERAL}/Fantastik_Break_Procesado_v2', 'Datos/fantastic_breaks/procesado_v2'),
    'obj': (f'{BASE_GENERAL}/roturas_Objaverse_v2',          'Datos/objaverse/roturas_v2'),
    'sn':  (f'{BASE_GENERAL}/shapenet_roturas',               'Datos/shapenet/roturas'),
}
FUENTES_ACTIVAS = {k: v for k, v in FUENTES.items() if k in _partes}

for clave, (drive_path, local_path) in FUENTES_ACTIVAS.items():
    dst = Path(local_path)
    if dst.exists() and any(dst.glob('*.npy')):
        print(f'[OK] {clave}: {len(list(dst.glob("*.npy")))} npy')
    elif Path(drive_path).exists():
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync','-a','--no-links',f'{drive_path}/',str(dst)], capture_output=True)
        print(f'[OK] {clave}: copiado {len(list(dst.glob("*.npy")))} npy')
    else:
        print(f'[WARN] {clave}: no encontrado')

from E3.dataset import construir_pares, ShapeCompletionDataset
import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False

carpetas = [v[1] for v in FUENTES_ACTIVAS.values() if Path(v[1]).exists()]
_todos = construir_pares(carpetas)
_rng = random.Random(42); _rng.shuffle(_todos)
_n = len(_todos); _nt = int(0.8*_n); _nv = int(0.1*_n)
_pares_test = _todos[_nt+_nv:]
print(f'Test: {len(_pares_test)} pares')

if ALEATORIO:
    pares_viz = random.sample(_pares_test, min(N_MOSTRAR or len(_pares_test), len(_pares_test)))
else:
    pares_viz = _pares_test[:N_MOSTRAR or len(_pares_test)]
print(f'Mostrando: {len(pares_viz)}')

In [ ]:
# ── CELDA 6: Visualizaciones ───────────────────────────────────
import matplotlib
matplotlib.use('Agg')  # renderiza a PNG aunque no haya display
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D  # noqa
import numpy as np, torch
from pathlib import Path

vis_dir = Path(f'E3/visualizaciones_mpl_{VERSION}')
vis_dir.mkdir(parents=True, exist_ok=True)

def inferir(roto_np):
    with torch.no_grad():
        inp = torch.tensor(roto_np).unsqueeze(0).to(device)
        out = model(inp)
        return (out[-1] if isinstance(out,(list,tuple)) else out).squeeze(0).cpu().numpy()

def cd_np(p, g):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    d = torch.cdist(pt, gt, p=2)
    return ((d.min(2).values.mean() + d.min(1).values.mean()) / 2).item()

def fs_np(p, g, th=0.01):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    dpg = torch.cdist(pt, gt, p=2); dgp = torch.cdist(gt, pt, p=2)
    pr = (dpg.min(2).values < th).float().mean()
    re = (dgp.min(2).values < th).float().mean()
    if pr + re < 1e-8: return 0.0
    return (2*pr*re/(pr+re)).item()

def error_pp(p, g):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    return torch.cdist(pt, gt, p=2).min(2).values.squeeze(0).numpy()

def scatter3(ax, pts, color, s=1, alpha=0.6, label=None, c_arr=None, cmap=None, vmin=None, vmax=None):
    kw = dict(s=s, alpha=alpha, depthshade=True, label=label)
    if c_arr is not None:
        sc = ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=c_arr, cmap=cmap,
                        vmin=vmin, vmax=vmax, **kw)
        return sc
    else:
        ax.scatter(pts[:,0], pts[:,1], pts[:,2], color=color, **kw)
        return None

def estilo_ax(ax, titulo, az=ANGULO_AZ, el=ANGULO_EL):
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.set_title(titulo, fontsize=9, pad=4)
    ax.view_init(elev=el, azim=az)
    ax.set_box_aspect([1,1,1])

print(f'Generando {len(pares_viz)} figuras...\n')

for idx_v, (ruta_r, ruta_c) in enumerate(pares_viz):
    roto_np = np.load(ruta_r).astype(np.float32)
    comp_np  = np.load(ruta_c).astype(np.float32)
    pred_np  = inferir(roto_np)
    nombre   = Path(ruta_r).stem.replace('_roto', '')
    dataset  = 'FB' if nombre.startswith('00_') or nombre.startswith('01_') else 'Obj'

    cd  = cd_np(pred_np, comp_np)
    fs  = fs_np(pred_np, comp_np)
    err = error_pp(pred_np, comp_np)
    err_max = np.percentile(err, 95)  # clip al p95 para que los outliers no dominen el colormap

    fig = plt.figure(figsize=(18, 4.5))
    fig.patch.set_facecolor('#0F1117')

    axes = [fig.add_subplot(1, 4, i+1, projection='3d') for i in range(4)]
    for ax in axes:
        ax.set_facecolor('#0F1117')
        ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
        ax.grid(False)

    # Panel 1: solo roto
    scatter3(axes[0], roto_np,  '#EF5350', s=3, alpha=0.9)
    estilo_ax(axes[0], 'Entrada (roto)')

    # Panel 2: roto + GT — qué falta
    scatter3(axes[1], comp_np, '#42A5F5', s=1, alpha=0.18, label='GT')
    scatter3(axes[1], roto_np, '#EF5350', s=3, alpha=0.9,  label='roto')
    estilo_ax(axes[1], 'Roto sobre GT')

    # Panel 3: predicción + GT — calidad reconstrucción
    scatter3(axes[2], comp_np, '#42A5F5', s=1, alpha=0.18, label='GT')
    scatter3(axes[2], pred_np, '#66BB6A', s=2, alpha=0.85, label='pred')
    estilo_ax(axes[2], 'Predicción sobre GT')

    # Panel 4: predicción coloreada por error
    sc = scatter3(axes[3], pred_np, None, s=2, alpha=0.9,
                  c_arr=err, cmap='RdYlGn_r', vmin=0, vmax=err_max)
    estilo_ax(axes[3], 'Error por punto (verde=bien, rojo=mal)')
    if sc is not None:
        cbar = fig.colorbar(sc, ax=axes[3], pad=0.0, shrink=0.6, aspect=15)
        cbar.ax.yaxis.set_tick_params(color='white', labelsize=7)
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
        cbar.set_label('dist', color='white', fontsize=7)

    # Título global
    titulo = f'[{idx_v+1}/{len(pares_viz)}]  {nombre}  ({dataset})   CD={cd:.4f}   F-Score={fs:.4f}   Modelo: PoinTr {VERSION}'
    fig.suptitle(titulo, color='white', fontsize=9, y=1.01)

    plt.tight_layout(pad=0.5)
    img_path = vis_dir / f'{idx_v:03d}_{nombre[:40]}_CD{cd:.4f}.png'
    plt.savefig(img_path, dpi=120, bbox_inches='tight', facecolor='#0F1117')
    plt.show()  # muestra inline en Colab
    plt.close()
    print(f'  [{idx_v+1}] {nombre[:50]:50s}  CD={cd:.4f}  F={fs:.4f}  dataset={dataset}')

print(f'\nFiguras guardadas en: {vis_dir}')

In [ ]:
# ── CELDA 7 (extra): Cuadrícula resumen — todas las muestras en una figura
# Muestra solo el panel 3 (predicción+GT) de cada muestra, para ver de un vistazo.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, torch, math
from pathlib import Path

n = len(pares_viz)
cols = min(4, n)
rows = math.ceil(n / cols)
fig = plt.figure(figsize=(cols * 4, rows * 4))
fig.patch.set_facecolor('#0F1117')

for idx_v, (ruta_r, ruta_c) in enumerate(pares_viz):
    roto_np = np.load(ruta_r).astype(np.float32)
    comp_np  = np.load(ruta_c).astype(np.float32)
    pred_np  = inferir(roto_np)
    nombre   = Path(ruta_r).stem.replace('_roto', '')
    dataset  = 'FB' if nombre.startswith('00_') or nombre.startswith('01_') else 'Obj'
    cd = cd_np(pred_np, comp_np)
    fs = fs_np(pred_np, comp_np)

    ax = fig.add_subplot(rows, cols, idx_v+1, projection='3d')
    ax.set_facecolor('#0F1117')
    ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
    ax.grid(False)
    ax.scatter(comp_np[:,0], comp_np[:,1], comp_np[:,2], s=0.5, color='#42A5F5', alpha=0.15, depthshade=True)
    ax.scatter(pred_np[:,0], pred_np[:,1], pred_np[:,2], s=1.5, color='#66BB6A', alpha=0.8, depthshade=True)
    ax.set_xlim(-1.1,1.1); ax.set_ylim(-1.1,1.1); ax.set_zlim(-1.1,1.1)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.view_init(elev=ANGULO_EL, azim=ANGULO_AZ)
    ax.set_box_aspect([1,1,1])
    color_titulo = '#A5D6A7' if cd < 0.035 else ('#FFF176' if cd < 0.05 else '#EF9A9A')
    ax.set_title(f'{dataset} CD={cd:.3f} F={fs:.3f}', color=color_titulo, fontsize=7, pad=2)

fig.suptitle(f'Cuadrícula resumen — verde=predicción, azul=GT | PoinTr {VERSION}',
             color='white', fontsize=10, y=1.01)
plt.tight_layout(pad=0.3)
grid_path = vis_dir / f'resumen_grid_{VERSION}.png'
plt.savefig(grid_path, dpi=120, bbox_inches='tight', facecolor='#0F1117')
plt.show()
plt.close()
print(f'Grid guardado: {grid_path}')
print('Título verde=CD<0.035 (bueno) | amarillo=CD<0.05 | rojo=CD>=0.05 (malo)')

In [ ]:
# ── CELDA 8 (opcional): Copiar imágenes a Drive ─────────────────
import shutil
from pathlib import Path

dst = Path(f'{BASE_E3}/visualizaciones/{VERSION}')
dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(vis_dir), str(dst), dirs_exist_ok=True)
print(f'Copiado a Drive: {dst}')
print(f'Archivos: {len(list(dst.glob("*.png")))}')